# 02 — Noise Synthesis
Generate OCR, ASR, and social media noisy versions of CoNLL-2003.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import random
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from noise import apply_ocr_noise, apply_asr_noise, apply_social_noise

In [2]:
dataset = load_dataset("conll2003", trust_remote_code=True)
label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
print(f"Labels: {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 14041 | Val: 3250 | Test: 3453


In [3]:
NOISE_SEED = 42

def apply_noise_to_split(split, noise_fn, p=0.3):
    """Apply noise_fn to every example in a dataset split. Returns list of dicts."""
    noisy_examples = []
    for ex in split:
        tokens = list(ex["tokens"])
        tags = list(ex["ner_tags"])
        noisy_tokens, noisy_tags = noise_fn(tokens, tags, id2label, label2id, p=p, seed=None)
        noisy_examples.append({
            "tokens": noisy_tokens,
            "ner_tags": noisy_tags,
        })
    return noisy_examples

def build_noisy_dataset(dataset, noise_fn, p=0.3):
    """Build a DatasetDict with train/validation/test splits."""
    random.seed(NOISE_SEED)
    splits = {}
    for split_name in ["train", "validation", "test"]:
        examples = apply_noise_to_split(dataset[split_name], noise_fn, p=p)
        splits[split_name] = Dataset.from_list(examples)
    return DatasetDict(splits)

In [4]:
print("Generating OCR noisy dataset...")
ocr_dataset = build_noisy_dataset(dataset, apply_ocr_noise)
print("Generating ASR noisy dataset...")
asr_dataset = build_noisy_dataset(dataset, apply_asr_noise)
print("Generating Social media noisy dataset...")
social_dataset = build_noisy_dataset(dataset, apply_social_noise)
print("Done.")

Generating OCR noisy dataset...
Generating ASR noisy dataset...
Generating Social media noisy dataset...
Done.


In [5]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "noisy")

ocr_dataset.save_to_disk(os.path.join(DATA_DIR, "ocr"))
asr_dataset.save_to_disk(os.path.join(DATA_DIR, "asr"))
social_dataset.save_to_disk(os.path.join(DATA_DIR, "social"))
print(f"Saved to {DATA_DIR}/{{ocr,asr,social}}/")

Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14041 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3250 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3453 [00:00<?, ? examples/s]

Saved to /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/data/noisy/{ocr,asr,social}/


## Examples: Before vs After

In [6]:
def show_examples(orig_split, noisy_split, n=3, title=""):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for i in range(n):
        orig = orig_split[i]
        noisy = noisy_split[i]
        orig_pairs = list(zip(orig["tokens"], [label_names[t] for t in orig["ner_tags"]]))
        noisy_pairs = list(zip(noisy["tokens"], [label_names[t] for t in noisy["ner_tags"]]))
        print(f"\nExample {i+1}:")
        print(f"  ORIG : {' '.join(f'{t}/{g}' for t,g in orig_pairs[:12])}")
        print(f"  NOISY: {' '.join(f'{t}/{g}' for t,g in noisy_pairs[:12])}")

show_examples(dataset["test"], ocr_dataset["test"], title="OCR Noise")
show_examples(dataset["test"], asr_dataset["test"], title="ASR Noise")
show_examples(dataset["test"], social_dataset["test"], title="Social Media Noise")


  OCR Noise

Example 1:
  ORIG : SOCCER/O -/O JAPAN/B-LOC GET/O LUCKY/O WIN/O ,/O CHINA/B-PER IN/O SURPRISE/O DEFEAT/O ./O
  NOISY: 5OCC3R/O -/O JAPAN/B-LOC GETLUCKY/O WIN/O ,/O CHINA/B-PER IN/O SURPRISEDEFEAT/O ./O

Example 2:
  ORIG : Nadim/B-PER Ladki/I-PER
  NOISY: N@d1m/B-PER L@dki/I-PER

Example 3:
  ORIG : AL-AIN/B-LOC ,/O United/B-LOC Arab/I-LOC Emirates/I-LOC 1996-12-06/O
  NOISY: AL-AIN/B-LOC ,/O United/B-LOC Arab/I-LOC Emirates/I-LOC 1996-12-06/O

  ASR Noise

Example 1:
  ORIG : SOCCER/O -/O JAPAN/B-LOC GET/O LUCKY/O WIN/O ,/O CHINA/B-PER IN/O SURPRISE/O DEFEAT/O ./O
  NOISY: soccer/O japan/B-LOC get/O lucky/O win/O china/B-PER in/O surprise/O defeat/O

Example 2:
  ORIG : Nadim/B-PER Ladki/I-PER
  NOISY: nadim/B-PER ladki/I-PER

Example 3:
  ORIG : AL-AIN/B-LOC ,/O United/B-LOC Arab/I-LOC Emirates/I-LOC 1996-12-06/O
  NOISY: al-ain/B-LOC united/B-LOC arab/I-LOC emirates/I-LOC 1996-12-06/O

  Social Media Noise

Example 1:
  ORIG : SOCCER/O -/O JAPAN/B-LOC GET/O LUCKY/O WI

## Statistics

In [7]:
def compute_stats(orig_split, noisy_split, name):
    orig_lens = [len(ex["tokens"]) for ex in orig_split]
    noisy_lens = [len(ex["tokens"]) for ex in noisy_split]

    changed = 0
    total_tokens = 0
    for orig, noisy in zip(orig_split, noisy_split):
        orig_toks = orig["tokens"]
        noisy_toks = noisy["tokens"]
        total_tokens += len(orig_toks)  # count ALL original tokens
        for j, ot in enumerate(orig_toks):
            # deleted tokens (noisy is shorter) count as changed
            if j >= len(noisy_toks) or ot.lower() != noisy_toks[j].lower():
                changed += 1

    return {
        "noise_type": name,
        "avg_orig_len": round(sum(orig_lens)/len(orig_lens), 2),
        "avg_noisy_len": round(sum(noisy_lens)/len(noisy_lens), 2),
        "pct_tokens_changed": round(100 * changed / max(total_tokens, 1), 2),
    }

stats = [
    compute_stats(dataset["test"], ocr_dataset["test"], "OCR"),
    compute_stats(dataset["test"], asr_dataset["test"], "ASR"),
    compute_stats(dataset["test"], social_dataset["test"], "Social"),
]
pd.DataFrame(stats)

,noise_type,avg_orig_len,avg_noisy_len,pct_tokens_changed
0,OCR,13.45,12.91,50.41
1,ASR,13.45,11.47,57.83
2,Social,13.45,13.45,25.19


## Tokenizer Stats on Noisy Data
Fertility, OOV rate, and UNK rate for each tokenizer on clean vs noisy text.

In [8]:
from transformers import AutoTokenizer
from metrics import compute_tokenizer_stats

MODEL_NAMES = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "google/byt5-small",
]

def load_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

# Gather sentence lists for each split
splits = {
    "clean":  [ex["tokens"] for ex in dataset["test"]],
    "ocr":    [ex["tokens"] for ex in ocr_dataset["test"]],
    "asr":    [ex["tokens"] for ex in asr_dataset["test"]],
    "social": [ex["tokens"] for ex in social_dataset["test"]],
}

rows = []
for model_name in MODEL_NAMES:
    print(f"Analyzing {model_name}...")
    tok = load_tokenizer(model_name)
    for split_name, sentences in splits.items():
        stats = compute_tokenizer_stats(tok, sentences, sample_size=len(sentences))
        rows.append({
            "model": model_name,
            "split": split_name,
            "fertility": stats["fertility"],
            "oov_rate": stats["oov_rate"],
            "unk_rate": stats["unk_rate"],
        })

tok_stats_df = pd.DataFrame(rows)

# Pivot for readability: model x split -> fertility
print("\n=== Fertility ===")
print(tok_stats_df.pivot(index="model", columns="split", values="fertility").round(3).to_string())
print("\n=== OOV Rate ===")
print(tok_stats_df.pivot(index="model", columns="split", values="oov_rate").round(4).to_string())
print("\n=== UNK Rate ===")
print(tok_stats_df.pivot(index="model", columns="split", values="unk_rate").round(4).to_string())

Analyzing bert-base-uncased...
Analyzing bert-base-cased...
Analyzing gpt2...
Analyzing google/byt5-small...

=== Fertility ===
split                asr  clean    ocr  social
model                                         
bert-base-cased    1.523  1.367  2.005   1.862
bert-base-uncased  1.289  1.247  1.867   1.626
google/byt5-small  4.845  4.310  4.500   4.410
gpt2               1.426  1.299  1.880   1.710

=== OOV Rate ===
split              asr  clean  ocr  social
model                                     
bert-base-cased    0.0    0.0  0.0     0.0
bert-base-uncased  0.0    0.0  0.0     0.0
google/byt5-small  0.0    0.0  0.0     0.0
gpt2               0.0    0.0  0.0     0.0

=== UNK Rate ===
split              asr  clean  ocr  social
model                                     
bert-base-cased    0.0    0.0  0.0     0.0
bert-base-uncased  0.0    0.0  0.0     0.0
google/byt5-small  0.0    0.0  0.0     0.0
gpt2               0.0    0.0  0.0     0.0


In [9]:
# Save tokenizer stats on noisy data
tok_stats_df.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "tables", "tokenizer_stats_noisy.csv"), index=False)
print("Saved tokenizer_stats_noisy.csv")
tok_stats_df

Saved tokenizer_stats_noisy.csv


,model,split,fertility,oov_rate,unk_rate
0,bert-base-uncased,clean,1.247,0.0,0.0
1,bert-base-uncased,ocr,1.867,0.0,0.0
2,bert-base-uncased,asr,1.289,0.0,0.0
3,bert-base-uncased,social,1.626,0.0,0.0
4,bert-base-cased,clean,1.367,0.0,0.0
5,bert-base-cased,ocr,2.005,0.0,0.0
6,bert-base-cased,asr,1.523,0.0,0.0
7,bert-base-cased,social,1.862,0.0,0.0
8,gpt2,clean,1.299,0.0,0.0
9,gpt2,ocr,1.880,0.0,0.0
